# Lecture 8: Gram-Schmidt Orthogonalization
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

## Classical Gram-Schmidt (CGS) — Step by Step

Orthogonalize $a_1 = (1,1,0)^T$, $a_2 = (1,0,1)^T$, $a_3 = (0,1,1)^T$.

In [ ]:
A = np.array([[1, 1, 0],
              [1, 0, 1],
              [0, 1, 1]], dtype=float).T  # columns are a1, a2, a3
# Actually we want columns:
A = np.array([[1, 1, 0],
              [1, 0, 1],
              [0, 1, 1]], dtype=float)

m, n = A.shape
Q = np.zeros((m, n))
R = np.zeros((n, n))

# Step 1
R[0, 0] = np.linalg.norm(A[:, 0])
Q[:, 0] = A[:, 0] / R[0, 0]
print(f"Step 1: q1 = {Q[:, 0].round(6)}")

# Step 2
R[0, 1] = Q[:, 0] @ A[:, 1]
v2 = A[:, 1] - R[0, 1] * Q[:, 0]
R[1, 1] = np.linalg.norm(v2)
Q[:, 1] = v2 / R[1, 1]
print(f"Step 2: q2 = {Q[:, 1].round(6)}")

# Step 3
R[0, 2] = Q[:, 0] @ A[:, 2]
R[1, 2] = Q[:, 1] @ A[:, 2]
v3 = A[:, 2] - R[0, 2] * Q[:, 0] - R[1, 2] * Q[:, 1]
R[2, 2] = np.linalg.norm(v3)
Q[:, 2] = v3 / R[2, 2]
print(f"Step 3: q3 = {Q[:, 2].round(6)}")

print(f"\nOrthogonality check:")
print(f"  q1·q2 = {Q[:, 0] @ Q[:, 1]:.2e}")
print(f"  q1·q3 = {Q[:, 0] @ Q[:, 2]:.2e}")
print(f"  q2·q3 = {Q[:, 1] @ Q[:, 2]:.2e}")
print(f"||QR - A|| = {np.linalg.norm(Q @ R - A):.2e}")

## CGS and MGS Implementations

In [ ]:
def classical_gram_schmidt(A):
    """Classical Gram-Schmidt: project against original a_j."""
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]   # project against ORIGINAL a_j
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

def modified_gram_schmidt(A):
    """Modified Gram-Schmidt: project against updated v."""
    m, n = A.shape
    V = A.copy()
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        R[j, j] = np.linalg.norm(V[:, j])
        Q[:, j] = V[:, j] / R[j, j]
        for k in range(j + 1, n):
            R[j, k] = Q[:, j] @ V[:, k]   # project against UPDATED v_k
            V[:, k] -= R[j, k] * Q[:, j]
    return Q, R

# Quick test on a well-conditioned matrix
np.random.seed(0)
A = np.random.randn(6, 4)

Q_cgs, R_cgs = classical_gram_schmidt(A)
Q_mgs, R_mgs = modified_gram_schmidt(A)

print("Well-conditioned matrix:")
print(f"  CGS: ||Q^TQ - I|| = {np.linalg.norm(Q_cgs.T @ Q_cgs - np.eye(4)):.2e}, ||A-QR|| = {np.linalg.norm(A - Q_cgs @ R_cgs):.2e}")
print(f"  MGS: ||Q^TQ - I|| = {np.linalg.norm(Q_mgs.T @ Q_mgs - np.eye(4)):.2e}, ||A-QR|| = {np.linalg.norm(A - Q_mgs @ R_mgs):.2e}")

## The Numerical Disaster: Nearly Dependent Columns

With $\epsilon = 10^{-8}$ and nearly parallel columns, CGS loses orthogonality catastrophically.

In [ ]:
eps = 1e-8
A = np.array([[1, 1,   1  ],
              [0, eps, 0  ],
              [0, 0,   eps]])

print(f"Condition number: {np.linalg.cond(A):.2e}\n")

Q_cgs, R_cgs = classical_gram_schmidt(A)
Q_mgs, R_mgs = modified_gram_schmidt(A)

print("Classical Gram-Schmidt:")
print(f"  ||Q^TQ - I|| = {np.linalg.norm(Q_cgs.T @ Q_cgs - np.eye(3)):.6e}")
print(f"  ||A - QR||   = {np.linalg.norm(A - Q_cgs @ R_cgs):.6e}")
print(f"  q1·q3        = {Q_cgs[:, 0] @ Q_cgs[:, 2]:.6e}")

print("\nModified Gram-Schmidt:")
print(f"  ||Q^TQ - I|| = {np.linalg.norm(Q_mgs.T @ Q_mgs - np.eye(3)):.6e}")
print(f"  ||A - QR||   = {np.linalg.norm(A - Q_mgs @ R_mgs):.6e}")
print(f"  q1·q3        = {Q_mgs[:, 0] @ Q_mgs[:, 2]:.6e}")

# NumPy (Householder)
Q_np, R_np = np.linalg.qr(A)
print("\nNumPy QR (Householder):")
print(f"  ||Q^TQ - I|| = {np.linalg.norm(Q_np.T @ Q_np - np.eye(3)):.6e}")
print(f"  ||A - QR||   = {np.linalg.norm(A - Q_np @ R_np):.6e}")

## Orthogonality Loss vs. Condition Number

Sweep over matrices with increasing condition number and measure $\|Q^TQ - I\|$.

In [ ]:
np.random.seed(42)
m, n = 50, 10
kappas = np.logspace(1, 14, 20)

orth_cgs = []
orth_mgs = []
orth_hh  = []

for kappa in kappas:
    # Build a matrix with prescribed condition number
    U, _ = np.linalg.qr(np.random.randn(m, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -np.log10(kappa), n)
    A = U @ np.diag(s) @ V.T

    Q_c, R_c = classical_gram_schmidt(A)
    Q_m, R_m = modified_gram_schmidt(A)
    Q_h, R_h = np.linalg.qr(A)

    orth_cgs.append(np.linalg.norm(Q_c.T @ Q_c - np.eye(n)))
    orth_mgs.append(np.linalg.norm(Q_m.T @ Q_m - np.eye(n)))
    orth_hh.append(np.linalg.norm(Q_h.T @ Q_h - np.eye(n)))

eps_mach = np.finfo(float).eps

plt.figure(figsize=(8, 5))
plt.loglog(kappas, orth_cgs, 'ro-', markersize=5, label='CGS')
plt.loglog(kappas, orth_mgs, 'bs-', markersize=5, label='MGS')
plt.loglog(kappas, orth_hh, 'g^-', markersize=5, label='Householder (NumPy)')
plt.loglog(kappas, kappas * eps_mach, 'k--', alpha=0.5, label=r'$\kappa \cdot \epsilon_{\mathrm{machine}}$')
plt.axhline(1, color='gray', linestyle=':', alpha=0.5, label='Total loss')
plt.xlabel(r'Condition number $\kappa$')
plt.ylabel(r'$\|Q^TQ - I\|$')
plt.title('Loss of orthogonality: CGS vs MGS vs Householder')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## The Projector Viewpoint

The projection $P_j = I - \hat{Q}_j \hat{Q}_j^T$ can be written as a sum or product of rank-1 projectors.

In [ ]:
np.random.seed(1)
Qj, _ = np.linalg.qr(np.random.randn(6, 3))
Qj = Qj[:, :3]

# Full projector onto orthogonal complement
Pj = np.eye(6) - Qj @ Qj.T
print(f"Singular values of Pj: {np.linalg.svd(Pj, compute_uv=False).round(10)}")

# Sum of rank-1 projections
Mj = np.eye(6)
for k in range(3):
    Mj -= np.outer(Qj[:, k], Qj[:, k])
print(f"||Pj - Mj|| (sum form):     {np.linalg.norm(Pj - Mj):.2e}")

# Product of rank-1 complementary projectors
Gj = np.eye(6)
for k in range(3):
    Gj = Gj @ (np.eye(6) - np.outer(Qj[:, k], Qj[:, k]))
print(f"||Pj - Gj|| (product form): {np.linalg.norm(Pj - Gj):.2e}")

## Residual $\|A - QR\|$: MGS Wins Even When Orthogonality Is Lost

In [ ]:
np.random.seed(7)
m, n = 50, 10
kappas = np.logspace(1, 14, 20)

res_cgs = []
res_mgs = []

for kappa in kappas:
    U, _ = np.linalg.qr(np.random.randn(m, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -np.log10(kappa), n)
    A = U @ np.diag(s) @ V.T

    Q_c, R_c = classical_gram_schmidt(A)
    Q_m, R_m = modified_gram_schmidt(A)

    res_cgs.append(np.linalg.norm(A - Q_c @ R_c) / np.linalg.norm(A))
    res_mgs.append(np.linalg.norm(A - Q_m @ R_m) / np.linalg.norm(A))

plt.figure(figsize=(8, 5))
plt.loglog(kappas, res_cgs, 'ro-', markersize=5, label='CGS')
plt.loglog(kappas, res_mgs, 'bs-', markersize=5, label='MGS')
plt.axhline(eps_mach, color='gray', linestyle='--', alpha=0.5, label=r'$\epsilon_{\mathrm{machine}}$')
plt.xlabel(r'Condition number $\kappa$')
plt.ylabel(r'$\|A - QR\| / \|A\|$')
plt.title('Factorization residual: CGS vs MGS')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Operation Count: Timing vs. $O(mn^2)$

In [ ]:
m_fixed = 400
n_vals = np.arange(20, 201, 20)
times_cgs = []
times_mgs = []

for n in n_vals:
    A = np.random.randn(m_fixed, n)

    t0 = time.perf_counter()
    _ = classical_gram_schmidt(A)
    times_cgs.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    _ = modified_gram_schmidt(A)
    times_mgs.append(time.perf_counter() - t0)

# Fit a reference n^2 curve
ref = (n_vals / n_vals[-1])**2 * times_cgs[-1]

plt.figure(figsize=(7, 4.5))
plt.loglog(n_vals, times_cgs, 'ro-', markersize=5, label='CGS')
plt.loglog(n_vals, times_mgs, 'bs-', markersize=5, label='MGS')
plt.loglog(n_vals, ref, 'k--', alpha=0.5, label=r'$O(n^2)$ reference')
plt.xlabel('$n$ (number of columns)')
plt.ylabel('Elapsed time (s)')
plt.title(f'Gram-Schmidt timing (m = {m_fixed})')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Visualizing CGS vs MGS on a 2D Example

Show how CGS and MGS differ when orthogonalizing nearly parallel vectors.

In [ ]:
# 3 nearly parallel vectors in R^2 (exaggerated for visibility)
theta = 0.08  # small angle
a1 = np.array([1.0, 0.0])
a2 = np.array([np.cos(theta), np.sin(theta)])
a3 = np.array([np.cos(-theta/2), np.sin(-theta/2)])

# CGS
A = np.column_stack([a1, a2, a3])
# For 2D we can only orthogonalize 2 vectors; show 2 steps
A2 = np.column_stack([a1, a2])

Q_c, R_c = classical_gram_schmidt(A2)
Q_m, R_m = modified_gram_schmidt(A2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, Q, title in zip(axes, [Q_c, Q_m], ['Classical GS', 'Modified GS']):
    # Original vectors
    ax.annotate('', xy=a1, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    ax.annotate('', xy=a2, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.text(a1[0]+0.02, a1[1]+0.03, '$a_1$', fontsize=12, color='blue')
    ax.text(a2[0]+0.02, a2[1]+0.03, '$a_2$', fontsize=12, color='red')

    # Orthonormal vectors (scaled for visibility)
    s = 0.5
    ax.annotate('', xy=s*Q[:, 0], xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
    ax.annotate('', xy=s*Q[:, 1], xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='darkgreen', lw=2.5))
    ax.text(s*Q[0, 0]-0.1, s*Q[1, 0]+0.03, '$q_1$', fontsize=12, color='green')
    ax.text(s*Q[0, 1]-0.1, s*Q[1, 1]+0.03, '$q_2$', fontsize=12, color='darkgreen')

    orth = Q[:, 0] @ Q[:, 1]
    ax.set_title(f'{title}\n$q_1 \\cdot q_2 = {orth:.2e}$', fontsize=11)
    ax.set_xlim(-0.2, 1.2)
    ax.set_ylim(-0.6, 0.6)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Gram-Schmidt on nearly parallel vectors', fontsize=13)
plt.tight_layout()
plt.show()

## Triangular Orthogonalization vs. Orthogonal Triangularization

Gram-Schmidt: $\hat{Q} = A \hat{R}^{-1}$ (triangular ops → orthogonal result)  
Householder: $\hat{R} = Q^T A$ (orthogonal ops → triangular result)

In [ ]:
np.random.seed(12)
A = np.random.randn(5, 3)

# Gram-Schmidt: Q = A R^{-1}
Q_gs, R_gs = modified_gram_schmidt(A)
R_inv = np.linalg.inv(R_gs)
Q_check = A @ R_inv
print("Gram-Schmidt: Q = A R^{-1}")
print(f"  ||Q_gs - A R^{{-1}}|| = {np.linalg.norm(Q_gs - Q_check):.2e}")

# Householder: R = Q^T A
Q_hh, R_hh = np.linalg.qr(A)
R_check = Q_hh.T @ A
print("\nHouseholder: R = Q^T A")
print(f"  ||R_hh - Q^T A|| = {np.linalg.norm(R_hh - R_check):.2e}")

print("\nGram-Schmidt applies triangular operations to A to get Q.")
print("Householder applies orthogonal operations to A to get R.")
print("Orthogonal operations preserve norms → Householder is more stable.")